# ImageNet minibatch experiments with full-dataset cosine

This is the ImageNet counterpart of `MNIST_UNO_UNOS_fixed_batch_cosine.ipynb`. It runs **Retraining**, **UNO**, **UNO-S**, and **Gradient Surgery** using the repository's shuffled minibatch processors. The only full-dataset computation is a read-only cosine diagnostic over every retain-class and forget-class image.

The experiment hyperparameters come from `ImageNet-Experiments-1.ipynb`. In particular, the attached notebook uses `orthogonality_weight = 0e-2`, which is exactly `0.0` in Python; that value is preserved verbatim here.

`Step = 0` is measured before training. Image generation uses the repository sampler's fixed seed (`42`), so each state and method is evaluated from the same latent noise.

In Colab, the setup installs only missing ImageNet-specific dependencies. It intentionally preserves Colab's preinstalled NumPy, SciPy, PyTorch, and torchvision binaries; installing the repository's full pinned `requirements.txt` into a running Colab kernel can create a mixed binary environment.

In [ ]:
import os
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

REPO_MARKER = Path("modules/imagenet/classifier.py")


def is_forget_repo(path: Path) -> bool:
    return (path / REPO_MARKER).is_file()


def find_existing_repo(candidates):
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate).expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if is_forget_repo(candidate):
            return candidate
    return None


def add_repo_modules_to_path(path: Path) -> None:
    shared_modules = path / "modules"
    imagenet_modules = shared_modules / "imagenet"
    if not (imagenet_modules / "classifier.py").is_file():
        raise FileNotFoundError(
            f"Invalid repository checkout at {path}: "
            "modules/imagenet/classifier.py is missing."
        )
    # Insert shared modules first so ImageNet-specific modules have final precedence.
    for module_dir in (shared_modules, imagenet_modules):
        module_dir_string = str(module_dir)
        while module_dir_string in sys.path:
            sys.path.remove(module_dir_string)
        sys.path.insert(0, module_dir_string)


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    content_root = Path("/content")
    working_directory = Path.cwd().resolve()
    repo_candidates = [
        content_root / "forget",
        content_root / "forget-repo",
        working_directory,
        *working_directory.parents,
    ]
    if content_root.is_dir():
        repo_candidates.extend(content_root.iterdir())

    repo_root = find_existing_repo(repo_candidates)
    if repo_root is None:
        clone_target = content_root / "forget"
        suffix = 1
        while clone_target.exists():
            clone_target = content_root / f"forget-repo-{suffix}"
            suffix += 1
        subprocess.run(
            ["git", "clone", "https://github.com/pinakm9/forget.git", str(clone_target)],
            check=True,
        )
        repo_root = clone_target

    if not is_forget_repo(repo_root):
        raise FileNotFoundError(
            f"Repository bootstrap failed: {repo_root / REPO_MARKER} was not found."
        )
    # Do not install the repository's full requirements.txt in Colab.
    # It pins NumPy, SciPy, PyTorch, and torchvision, which can replace
    # already-loaded Colab binaries and corrupt the live Python runtime.
    colab_dependencies = {
        "diffusers": "diffusers>=0.35,<0.36",
        "transformers": "transformers>=4.44,<5",
        "accelerate": "accelerate>=1.0",
        "timm": "timm>=1.0.19",
        "cleanfid": "clean-fid",
        "pytorch_msssim": "pytorch-msssim>=1.0",
        "einops": "einops>=0.8",
    }
    missing_dependencies = [
        requirement
        for module_name, requirement in colab_dependencies.items()
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_dependencies:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade-strategy",
                "only-if-needed",
                *missing_dependencies,
            ],
            check=True,
        )

    try:
        import numpy as _numpy_health_check
        import scipy as _scipy_health_check
    except Exception as dependency_error:
        raise RuntimeError(
            "Colab's NumPy/SciPy runtime is inconsistent, usually because "
            "the old setup cell installed the full requirements.txt. Use "
            "Runtime > Restart session, then run this updated notebook from "
            "the first cell."
        ) from dependency_error
    drive.mount("/content/drive")
    imagenet_root = Path("/content/drive/MyDrive/Pinak/forget/ImageNet")
    experiment_folder = Path("/content/ImageNet-Experiments")
else:
    working_directory = Path.cwd().resolve()
    repo_root = find_existing_repo([working_directory, *working_directory.parents])
    if repo_root is None:
        raise FileNotFoundError(
            "Could not find a repository containing modules/imagenet/classifier.py."
        )
    imagenet_root = repo_root / "data" / "ImageNet-1k"
    experiment_folder = repo_root / "data" / "ImageNet-Experiments"

add_repo_modules_to_path(repo_root)

model_path = imagenet_root / "DiT-XL-2"
imagenet_json_path = imagenet_root / "imagenet_class_index.json"
data_path = imagenet_root / "2012"

print(f"Repository: {repo_root}")
print(f"ImageNet root: {imagenet_root}")
print(f"Experiment folder: {experiment_folder}")

import gc
import json
import math
import time
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm.auto import tqdm

import classifier as cl
import generate as gn
import loss as ls

# Importing loss loads ImageNet's DiT wrapper, which also exposes fast-DiT.
# Reassert ImageNet path precedence and evict a cached fast-DiT train module.
add_repo_modules_to_path(repo_root)
expected_train_path = (repo_root / "modules" / "imagenet" / "train.py").resolve()
cached_train = sys.modules.get("train")
if cached_train is not None:
    cached_train_file = getattr(cached_train, "__file__", None)
    cached_train_path = (
        Path(cached_train_file).resolve() if cached_train_file else None
    )
    if cached_train_path != expected_train_path:
        del sys.modules["train"]

import train as tt
if Path(tt.__file__).resolve() != expected_train_path:
    raise ImportError(
        f"Resolved train from {tt.__file__}, expected {expected_train_path}."
    )

import ortho as uno
import save as sv
import surgery as surgery
import utility as ut

device = ut.get_device()
if torch.cuda.is_available():
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

print(f"Device: {device}")

## Hyperparameters from the attached Colab notebook

The training and generation values below are copied from `ImageNet-Experiments-1.ipynb`. `full_gradient_batch_size` only controls the memory footprint of the diagnostic and does not change minibatch training.

In [ ]:
num_experiments = 10

params = {
    "model_path": str(model_path),
    "num_steps": 100,
    "batch_size": 10,
    "log_interval": 1,
    "collect_interval": "epoch",
    "save_steps": None,
    "exchange_classes": [208],
    "forget_class": 207,
    "data_path": str(data_path),
    "imagenet_json_path": str(imagenet_json_path),
    "freeze_K": 4,
    "unfreeze_last": True,
    "keep_all": True,
    "n_samples": 50,
    "device": device,
    "diffusion_steps": 15,
    "learning_rate": 1e-4,
}

generation_kwargs = {
    "guidance_scale": 8.0,
    "n_steps": 15,
}

summary_generation_kwargs = {
    "guidance_scale": 2.5,
    "n_steps": 10,
}

fid_params = {
    "num_fid_samples": 22000,
    "fid_batch_size": 64,
}

full_gradient_batch_size = params["batch_size"]

method_specs = {
    "Retraining": {
        "folder": "dit-ret",
    },
    "UNO": {
        "folder": "dit-o",
        "orthogonality_weight": 0e-2,
    },
    "UNO-S": {
        "folder": "dit-os",
        "orthogonality_weight": 0e-2,
    },
    "Surgery": {
        "folder": "dit-s",
    },
}

required_paths = [model_path, imagenet_json_path, data_path]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing ImageNet assets: " + ", ".join(missing_paths))

experiment_folder.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "num_experiments": num_experiments,
    "params": {**params, "device": str(params["device"])},
    "generation_kwargs": generation_kwargs,
    "summary_generation_kwargs": summary_generation_kwargs,
    "fid_params": fid_params,
    "methods": method_specs,
}, indent=2))

## Read-only full-dataset cosine diagnostic

For the retain class, $g_r$ is the gradient of the same two terms used by ImageNet UNO: the correctly labelled retain loss plus the retain images relabelled as the forget class. For the forget class, $g_f$ is the ordinary forget-class diffusion loss gradient.

Every image contributes with sample-count weighting. Diffusion time, latent sampling, and noise remain stochastic, so this is a full-image-dataset Monte Carlo gradient rather than an analytic expectation over diffusion randomness. RNG states and module modes are restored before minibatch training continues.

Because the attached configuration has `keep_all=True`, one retain gradient is temporarily stored on CPU while the forget gradient is accumulated on the accelerator. This avoids keeping two full DiT-XL/2 gradient copies in accelerator memory.

In [ ]:
def capture_torch_rng_states(active_device):
    active_device = torch.device(active_device)
    states = {"cpu": torch.random.get_rng_state()}
    if torch.cuda.is_available():
        states["cuda"] = torch.cuda.get_rng_state_all()
    if active_device.type == "mps" and hasattr(torch.mps, "get_rng_state"):
        states["mps"] = torch.mps.get_rng_state()
    return states


def restore_torch_rng_states(states):
    torch.random.set_rng_state(states["cpu"])
    if "cuda" in states:
        torch.cuda.set_rng_state_all(states["cuda"])
    if "mps" in states:
        torch.mps.set_rng_state(states["mps"])


def capture_module_modes(*models):
    return [
        (module, module.training)
        for model in models
        for module in model.modules()
    ]


def restore_module_modes(module_modes):
    for module, was_training in module_modes:
        module.training = was_training


def diagnostic_autocast(active_device):
    active_device = torch.device(active_device)
    if active_device.type != "cuda":
        return nullcontext()
    capability = torch.cuda.get_device_capability(active_device)[0]
    amp_dtype = torch.bfloat16 if capability >= 8 else torch.float16
    return torch.autocast(device_type="cuda", dtype=amp_dtype)


def accumulate_full_gradient(
    model,
    vae,
    diffusion,
    loader,
    active_device,
    forget_class,
    split,
):
    """Accumulate a sample-weighted full-dataset gradient in parameter.grad."""
    model.zero_grad(set_to_none=True)
    total_samples = len(loader.dataset)

    for images, labels in loader:
        images = images.to(active_device)
        labels = labels.to(active_device)
        sample_weight = images.shape[0] / total_samples

        with diagnostic_autocast(active_device):
            if split == "retain":
                forget_labels = torch.full_like(labels, forget_class)
                relabelled_loss = ls.loss(
                    model, vae, diffusion, active_device, images, forget_labels
                )
                retain_loss = ls.loss(
                    model, vae, diffusion, active_device, images, labels
                )
                batch_loss = retain_loss + relabelled_loss
            elif split == "forget":
                batch_loss = ls.loss(
                    model,
                    vae,
                    diffusion,
                    active_device,
                    images,
                    labels,
                )
            else:
                raise ValueError(f"Unknown split: {split}")

        (sample_weight * batch_loss).backward()


def measure_full_dataset_cosine(
    model,
    vae,
    diffusion,
    retain_loader,
    forget_loader,
    trainable_params,
    active_device,
    forget_class,
):
    rng_states = capture_torch_rng_states(active_device)
    module_modes = capture_module_modes(model, vae)

    try:
        model.eval()
        vae.eval()

        accumulate_full_gradient(
            model,
            vae,
            diffusion,
            retain_loader,
            active_device,
            forget_class,
            "retain",
        )
        retain_gradients = [
            None if parameter.grad is None else parameter.grad.detach().cpu()
            for parameter in trainable_params
        ]
        retain_squared_norm = sum(
            torch.sum(gradient.square(), dtype=torch.float64)
            for gradient in retain_gradients
            if gradient is not None
        )

        accumulate_full_gradient(
            model,
            vae,
            diffusion,
            forget_loader,
            active_device,
            forget_class,
            "forget",
        )

        dot_product = torch.zeros((), dtype=torch.float64)
        forget_squared_norm = torch.zeros((), dtype=torch.float64)
        for retain_gradient, parameter in zip(retain_gradients, trainable_params):
            if parameter.grad is None:
                continue
            forget_gradient = parameter.grad.detach().cpu()
            forget_squared_norm += torch.sum(
                forget_gradient.square(),
                dtype=torch.float64,
            )
            if retain_gradient is not None:
                dot_product += torch.sum(
                    retain_gradient * forget_gradient,
                    dtype=torch.float64,
                )

        denominator = torch.sqrt(retain_squared_norm * forget_squared_norm).clamp_min(1e-24)
        cosine = (dot_product / denominator).clamp(-1.0, 1.0)
        result = {
            "Full Dataset Gradient Dot Product": float(dot_product),
            "Full Dataset Retain Gradient Norm": float(torch.sqrt(retain_squared_norm)),
            "Full Dataset Forget Gradient Norm": float(torch.sqrt(forget_squared_norm)),
            "Full Dataset Cosine Similarity": float(cosine),
            "Absolute Full Dataset Cosine Similarity": float(cosine.abs()),
            "Full Dataset Cosine Distance": float(1.0 - cosine),
        }
    finally:
        model.zero_grad(set_to_none=True)
        restore_module_modes(module_modes)
        restore_torch_rng_states(rng_states)

    return result


def generation_statistics(
    model,
    vae,
    identifier,
    active_device,
    forget_class,
    n_samples,
    generation_kwargs,
):
    rng_states = capture_torch_rng_states(active_device)
    module_modes = capture_module_modes(model, vae, identifier)
    try:
        model.eval()
        vae.eval()
        identifier.eval()
        labels = torch.full(
            (n_samples,),
            forget_class,
            dtype=torch.long,
            device=active_device,
        )
        generated_images = gn.generate(
            model,
            vae,
            labels,
            device=active_device,
            seed=42,
            **generation_kwargs,
        ).detach()
        forget_count = cl.identify(
            identifier,
            generated_images,
            forget_class,
            active_device,
        )
        forget_fraction = forget_count / generated_images.shape[0]
    finally:
        restore_module_modes(module_modes)
        restore_torch_rng_states(rng_states)

    return generated_images, float(forget_fraction)

## Instrumented minibatch trainer

Updates come directly from the repository processors: retain-only diffusion loss for Retraining, cosine-squared UNO, alternating UNO/Surgery for UNO-S, and projected gradients for Surgery. Retain and forget minibatches remain independently shuffled and are paired with `zip`, matching the ImageNet training files.

In [ ]:
def next_update_name(method, completed_steps):
    if completed_steps >= params["num_steps"]:
        return "None (final state)"
    if method == "UNO-S":
        return "UNO" if (completed_steps + 1) % 2 == 1 else "Surgery"
    return method


def train_instrumented_method(
    method,
    folder,
    full_gradient_batch_size,
    generation_kwargs,
    **cfg,
):
    if method not in method_specs:
        raise ValueError(f"Unknown method: {method}")

    folder = Path(folder)
    if folder.exists():
        shutil.rmtree(folder)

    orthogonality_weight = method_specs[method].get("orthogonality_weight")
    (
        model,
        vae,
        diffusion,
        dataloader,
        optimizer,
        z_random,
        identifier,
        sample_dir,
        checkpoint_dir,
        epoch_length,
        epochs,
        num_steps,
        save_steps,
        collect_interval,
        log_interval,
        csv_file,
        active_device,
        grid_size,
        trainable_params,
    ) = tt.init(
        model_path=cfg["model_path"],
        folder=str(folder),
        num_steps=cfg["num_steps"],
        batch_size=cfg["batch_size"],
        save_steps=cfg["save_steps"],
        collect_interval=cfg["collect_interval"],
        log_interval=cfg["log_interval"],
        learning_rate=cfg["learning_rate"],
        orthogonality_weight=orthogonality_weight,
        exchange_classes=cfg["exchange_classes"],
        forget_class=cfg["forget_class"],
        data_path=cfg["data_path"],
        imagenet_json_path=cfg["imagenet_json_path"],
        n_samples=cfg["n_samples"],
        device=cfg["device"],
        diffusion_steps=cfg["diffusion_steps"],
        freeze_K=cfg["freeze_K"],
        unfreeze_last=cfg["unfreeze_last"],
        keep_all=cfg["keep_all"],
    )

    tt.patch_checkpoint_nonreentrant()

    if method == "UNO":
        process_odd = uno.get_processor(
            model,
            vae,
            diffusion,
            active_device,
            optimizer,
            trainable_params,
            orthogonality_weight,
        )
        process_even = process_odd
    elif method == "UNO-S":
        process_odd = uno.get_processor(
            model,
            vae,
            diffusion,
            active_device,
            optimizer,
            trainable_params,
            orthogonality_weight,
        )
        process_even = surgery.get_processor(
            model,
            vae,
            diffusion,
            active_device,
            optimizer,
            trainable_params,
        )
    elif method == "Surgery":
        process_odd = surgery.get_processor(
            model,
            vae,
            diffusion,
            active_device,
            optimizer,
            trainable_params,
        )
        process_even = process_odd
    else:
        process_odd = tt.get_processor(
            model,
            vae,
            diffusion,
            active_device,
            optimizer,
        )
        process_even = process_odd

    full_retain_loader = DataLoader(
        dataloader["retain"].dataset,
        batch_size=full_gradient_batch_size,
        shuffle=False,
        drop_last=False,
    )
    full_forget_loader = DataLoader(
        dataloader["forget"].dataset,
        batch_size=full_gradient_batch_size,
        shuffle=False,
        drop_last=False,
    )

    rows = []
    previous_update = "None (initial state)"
    last_loss = float("nan")
    last_orthogonality = float("nan")
    last_update_time = 0.0

    def record_state(step):
        diagnostic = measure_full_dataset_cosine(
            model,
            vae,
            diffusion,
            full_retain_loader,
            full_forget_loader,
            trainable_params,
            active_device,
            cfg["forget_class"],
        )
        generated_images, forget_fraction = generation_statistics(
            model,
            vae,
            identifier,
            active_device,
            cfg["forget_class"],
            cfg["n_samples"],
            generation_kwargs,
        )

        if step == cfg["num_steps"]:
            final_sample_path = Path(sample_dir) / "sample_final.png"
            save_image(
                generated_images.cpu(),
                final_sample_path,
                nrow=max(1, int(math.sqrt(cfg["n_samples"]))),
            )

        row = {
            "Step": step,
            "Method": method,
            "Previous Update": previous_update,
            "Next Update": next_update_name(method, step),
            "Total Loss": last_loss,
            "Orthogonality Loss": last_orthogonality,
            "Time": last_update_time,
            "0 Fraction": 1.0 - forget_fraction,
            "1 Fraction": forget_fraction,
            **diagnostic,
        }
        rows.append(row)
        del generated_images

    record_state(0)

    global_step = 0
    progress = tqdm(total=cfg["num_steps"], desc=f"{method} minibatch updates")
    done = False
    for _ in range(epochs):
        paired_batches = zip(dataloader["retain"], dataloader["forget"])
        for batch_retain, batch_forget in paired_batches:
            if global_step >= cfg["num_steps"]:
                done = True
                break

            img_retain = batch_retain[0].to(active_device)
            label_retain = batch_retain[1].to(active_device)
            img_forget = batch_forget[0].to(active_device)
            label_forget = batch_forget[1].to(active_device)

            update_number = global_step + 1
            if method == "Retraining":
                last_loss, last_update_time = process_odd(
                    img_retain,
                    label_retain,
                )
                last_orthogonality = float("nan")
                previous_update = "Retraining"
            elif method == "UNO-S" and update_number % 2 == 0:
                last_loss, last_update_time = process_even(
                    img_retain,
                    label_retain,
                    img_forget,
                    label_forget,
                )
                last_orthogonality = float("nan")
                previous_update = "Surgery"
            elif method in {"UNO", "UNO-S"}:
                last_loss, last_update_time, last_orthogonality = process_odd(
                    img_retain,
                    label_retain,
                    img_forget,
                    label_forget,
                )
                previous_update = "UNO"
            else:
                last_loss, last_update_time = process_odd(
                    img_retain,
                    label_retain,
                    img_forget,
                    label_forget,
                )
                last_orthogonality = float("nan")
                previous_update = "Surgery"

            global_step = update_number
            record_state(global_step)
            progress.update(1)
            progress.set_postfix(
                full_abs_cos=f"{rows[-1]['Absolute Full Dataset Cosine Similarity']:.4f}",
                one_fraction=f"{rows[-1]['1 Fraction']:.4f}",
            )

        if done or global_step >= cfg["num_steps"]:
            break

    progress.close()
    if global_step != cfg["num_steps"]:
        raise RuntimeError(
            f"Expected {cfg['num_steps']} updates, completed {global_step}."
        )

    log = pd.DataFrame(rows)
    log.to_csv(Path(checkpoint_dir) / "training_log.csv", index=False)
    log[[
        "Step",
        "Full Dataset Gradient Dot Product",
        "Full Dataset Retain Gradient Norm",
        "Full Dataset Forget Gradient Norm",
        "Full Dataset Cosine Similarity",
        "Absolute Full Dataset Cosine Similarity",
        "Full Dataset Cosine Distance",
    ]].to_csv(Path(checkpoint_dir) / "full_dataset_cosine.csv", index=False)

    sv.save_trainable_checkpoint(
        model,
        str(Path(checkpoint_dir) / f"DiT_step_{global_step}.pth"),
    )

    config_path = folder / "config.json"
    with config_path.open() as config_file:
        config = json.load(config_file)
    config["diagnostic"] = {
        "full_gradient_batch_size": full_gradient_batch_size,
        "retain_samples": len(full_retain_loader.dataset),
        "forget_samples": len(full_forget_loader.dataset),
        "metric_timing": "Step t is the model after t minibatch updates",
        "generation_seed": 42,
        "generation_kwargs": generation_kwargs,
        "full_gradient_definition": "all images, one Monte Carlo diffusion draw per loss call",
    }
    with config_path.open("w") as config_file:
        json.dump(config, config_file, indent=2)

    del model, vae, diffusion, optimizer, identifier
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return log

## Run all methods

This is substantially more expensive than the attached UNO-only notebook: four methods × ten experiments are run, and every recorded state includes two complete class-dataset gradient passes plus fixed-seed generation. For a smoke test, temporarily reduce `num_experiments` or `num_steps`; restore the attached values for the actual experiment.

In [ ]:
logs = {}

for method, method_spec in method_specs.items():
    for experiment_id in range(num_experiments):
        run_folder = (
            experiment_folder
            / method_spec["folder"]
            / f"expr-{experiment_id}"
        )
        logs[(method, experiment_id)] = train_instrumented_method(
            method=method,
            folder=run_folder,
            full_gradient_batch_size=full_gradient_batch_size,
            generation_kwargs=generation_kwargs,
            **params,
        )

print(f"Results written to: {experiment_folder}")

## Plot full-dataset $|\cos(g_r,g_f)|$ and semilog `1 Fraction`

`1 Fraction` retains the ImageNet repository's column name: here it means the fraction of fixed-seed generated images classified as forget class 207. Curves show the mean across ten experiments with one-standard-deviation bands. Exact zero means are displayed at half of one generated sample and marked with ×.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
zero_display_floor = 0.5 / params["n_samples"]

for method in method_specs:
    method_frames = [
        logs[(method, experiment_id)].set_index("Step")
        for experiment_id in range(num_experiments)
    ]
    steps = method_frames[0].index.to_numpy()

    cosine_values = np.stack([
        frame["Absolute Full Dataset Cosine Similarity"].to_numpy()
        for frame in method_frames
    ])
    cosine_mean = cosine_values.mean(axis=0)
    cosine_std = cosine_values.std(axis=0)
    cosine_line = axes[0].plot(steps, cosine_mean, label=method)[0]
    axes[0].fill_between(
        steps,
        np.clip(cosine_mean - cosine_std, 0.0, 1.0),
        np.clip(cosine_mean + cosine_std, 0.0, 1.0),
        color=cosine_line.get_color(),
        alpha=0.18,
    )

    fraction_values = np.stack([
        frame["1 Fraction"].to_numpy()
        for frame in method_frames
    ])
    fraction_mean = fraction_values.mean(axis=0)
    fraction_std = fraction_values.std(axis=0)
    plotted_mean = np.where(fraction_mean == 0, zero_display_floor, fraction_mean)
    lower = np.maximum(fraction_mean - fraction_std, zero_display_floor)
    upper = np.maximum(fraction_mean + fraction_std, zero_display_floor)
    fraction_line = axes[1].semilogy(steps, plotted_mean, label=method)[0]
    axes[1].fill_between(
        steps,
        lower,
        upper,
        color=fraction_line.get_color(),
        alpha=0.18,
    )
    zero_mask = fraction_mean == 0
    if zero_mask.any():
        axes[1].scatter(
            steps[zero_mask],
            np.full(int(zero_mask.sum()), zero_display_floor),
            marker="x",
            color=fraction_line.get_color(),
            s=25,
        )

axes[0].set_title("Full-dataset absolute cosine similarity")
axes[0].set_xlabel("Completed minibatch updates")
axes[0].set_ylabel(r"$|\cos(g_r,g_f)|$")
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(alpha=0.3)

axes[1].axhline(
    0.02,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="0.02 threshold",
)
axes[1].set_title("Generated forget-class fraction")
axes[1].set_xlabel("Completed minibatch updates")
axes[1].set_ylabel("1 Fraction (log scale)")
axes[1].grid(alpha=0.3, which="both")

for axis in axes:
    axis.legend()

fig.tight_layout()
plot_path = experiment_folder / "full_dataset_absolute_cosine_and_one_fraction.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved plot: {plot_path}")

## Fixed-seed generated images after training

The following cell displays the final fixed-seed grid from experiment 0 for each method. Every run also saves its own `samples/sample_final.png`.

In [ ]:
from IPython.display import Image, display

for method, method_spec in method_specs.items():
    sample_path = (
        experiment_folder
        / method_spec["folder"]
        / "expr-0"
        / "samples"
        / "sample_final.png"
    )
    print(f"{method}: {sample_path}")
    display(Image(filename=str(sample_path)))

### Reading the results

- `training_log.csv` contains states 0 through 100. State $t$ is measured after exactly $t$ shuffled minibatch updates.
- `full_dataset_cosine.csv` contains the full class-dataset gradient diagnostic only.
- `1 Fraction` means the fraction classified as ImageNet class 207, not the MNIST digit 1.
- Generation is fixed across states through sampler seed 42; training minibatches and diffusion-loss randomness are not fixed.
- `orthogonality_weight=0e-2` comes directly from the attached notebook and therefore disables the orthogonality contribution even though UNO still constructs and reports it.
- The attached FID settings (`22000` samples, batch size `64`, guidance `2.5`, ten sampling steps) are retained in `fid_params` and `summary_generation_kwargs`, but FID is not run automatically in this already expensive notebook.